# Three-qubit system q1 – bus – q2 (Table 1 of the paper) with realistic physics

Chain **q1 – c – q2** and transfer $|1,0,0\rangle \to |0,0,1\rangle$: one excitation moves from qubit 1 to qubit 2 through the bus. The physics and the optimiser live in `floquet/three_qubit.py`, which reuses the kernels of `floquet/chain.py`.

**Table 1 of the original paper.**
- Parameters: $\omega_1=4.4$, $\omega_c=4.5$, $\omega_2=4.6$, $g=0.4$, delta kick $\lambda=4$, $T=10.95$, 447 cycles, $F=0.99999$.
- Issues raised by the referee:
  - the bus sits **between** the qubits and $g/\Delta=4$, so the modes are hybridised;
  - the detunings (100 MHz) are below the anharmonicity;
  - the kick is a delta (unphysical pulse);
  - no decoherence.

**Here (same rules as the 7-site Steps 2b/3).**
- Units: GHz and ns with $2\pi$; three-level transmon model ($\alpha=-0.2$ GHz).
- Physical constraints:
  - coupler above the qubits (≥ 0.7 GHz) and $g\le150$ MHz;
  - $\sigma\ge2$ ns and excursion $2A\le2$ GHz, with no coupler–qubit crossing during the kick;
  - gate ≤ 450 ns.
- The transfer without pulse must stay below `STATIC4['F_MAX']` (Floquet essential).
- The objective includes the relative noise $\eta=10^{-4}$ and decoherence.

**Search.** Phase G is global, with prescreening; phase L is a local refinement. It is interruptible: re-running the cell resumes from the checkpoint.

**Files.** Results in `results/step4`, analyses in `analysis/step4`, figures in `figures/step4` (relative to the repository root).

**Compute.** Everything runs on the CPU (D ≤ 27).

In [ ]:
%load_ext autoreload
%autoreload 2
import json, os, sys, math
sys.path.insert(0, os.path.abspath('..'))      # repository root (the notebook lives in notebooks/)
import numpy as np
import matplotlib.pyplot as plt
from floquet import chain as fv
from floquet import three_qubit as s4
s4.use_cpu()
print('device:', fv.DEVICE)

## Parameters

In [ ]:
# === STEP 4: parameters (editable) ===
s4.OPT4.update(
    ETA_OPT=1e-4,          # relative noise in the objective
    TIME_G=1800.0,         # s, global search
    TIME_L=900.0,          # s, local refinement
)
s4.STATIC4['F_MAX'] = 0.1  # tolerated F WITHOUT pulse (Floquet essential)
s4.PHYS4['TPROT_MAX'] = 450.0
print('OPT4   :', s4.OPT4)
print('STATIC4:', s4.STATIC4)

## Search + checks + analysis

In [ ]:
# === STEP 4: run (INTERRUPTIBLE: re-run the cell to resume) ===
REDO = False          # True: repeat the search even if the result exists (the old one becomes *_previous.json)
fn, fa = s4.path4(f'v5_step4_{s4.NAME}.json', 'res'), s4.path4(f'v5_step4_analysis_{s4.NAME}.json', 'an')
if os.path.exists(fn) and not REDO:
    o, an = json.load(open(fn)), json.load(open(fa))
    print('Existing result loaded (REDO = True to repeat the search)')
else:
    for p in (fn, fa):
        if os.path.exists(p):
            os.replace(p, p.replace('.json', '_previous.json'))
    o, an = s4.run_step4(fresh=REDO)

In [ ]:
# === Summary: Step 4 vs the original Table 1 ===
t1, a = an['original_table1'], o['audit']
eta_r = next(l for l in an['eta_scan']['levels'] if np.isclose(l['eta'], 1e-4))
rows = [
    ('F (qubit model)',                  f"{t1['qubit']['F_at_n_paper']:.5f}",        f"{o['fidelity_qubit_at_n_opt']:.5f}"),
    ('F (transmon, alpha = -0.2 GHz)',   f"{t1['qutrit']['F_at_n_paper']:.4f}",       f"{o['fidelity']:.5f}"),
    ('F with decoherence (Lindblad)',    '-',                                          f"{an['lindblad']['F_lindblad']:.5f}"),
    ('<F> with noise eta = 1e-4',        '-',                                          f"{eta_r['mean']:.4f}"),
    ('cycles n',                         f"{t1['paper']['n']}",                         f"{o['n_opt']}"),
    ('gate duration',                    f"~{t1['issues']['gate_time_if_GHz_ns']:.0f} ns (delta kick)", f"{o['protocol_time_ns']:.0f} ns"),
    ('bus position',                     'between the qubits',                          f"{a['min_Delta_qc_GHz']:.2f} GHz above"),
    ('g / Delta_qc',                     f"{t1['issues']['g_over_Delta_qc']:.1f}",       f"{a['max_g_over_Delta_qc']:.3f}"),
    ('|Delta_q1q2|',                     f"{t1['issues']['Delta_q1q2_vs_alpha'][0]*1e3:.0f} MHz", f"{a['Delta_q1q2_GHz']*1e3:.0f} MHz"),
    ('pulse',                            'delta, lambda = 4',                          f"Gaussian sigma = {o['sigma_opt']:.1f} ns, 2A = {2*o['A_opt']:.2f} GHz"),
    ('F without pulse (1.5 us)',         '-',                                          f"{o['F_static_long_window']:.3f}"),
]
print(f"{'':34s}{'original Table 1':>28s}   {'Step 4'}")
for lab, x0, x1 in rows:
    print(f"{lab:34s}{x0:>28s}   {x1}")
print('\nfrequencies (q1, c, q2) [GHz]:', np.round(o['omegas_opt'], 4).tolist(), '  g [GHz]:', np.round(o['couplings'], 4).tolist())
print(f"T = {o['T_opt']:.3f} ns, Floquet contribution C = {o['floquet_contribution']:.3f}, |2> leakage {o['leakage_qutrit_N4']:.1e}")

## Figures (same style as the 7-site notebook)

The cells are the same as in `seven_site_chain.ipynb`, adapted to the 3-site system q1 – c – q2 and to the three-level transmon model. The labels $|10\rangle$ and $|01\rangle$ denote the state of the two qubits (q1, q2); the bus starts and ends in $|0\rangle$.
- **Stroboscopic evolution**:
  - fidelity with and without pulse (A = 0, in grey, at the same times $t_n = n\,(T+6\sigma)$);
  - populations of the initial and final state.
- **Robustness curve**: $\bar F$ at fixed $n_{opt}$ and $\bar F_{\max}$ over $n$, with the top axis in MHz. Then $F$ as a function of $T_1$ and the $F(n)\pm$std bands.
- **Resolvent, FFT and $\Delta\varepsilon$ transitions** on the exact Floquet operator (full period $T + 6\sigma$).

The PDFs go to `figures/step4/` with the same names as for the 7-site chain (`v5_step4_Fidelity_10_01.pdf`, ...).

In [ ]:
# === Stroboscopic evolution, plots and data export ===
SAVE_PDF    = True
NPLOT       = 10000     # steps of the evolution (many points are needed for the FFT)
XMAX        = 150       # zoom of the plots
SHOW_STATIC = True      # overlay the fidelity without pulse (A = 0)
NAME, STEP = s4.NAME, 4
# load the saved solution and analysis: the figures do not require re-running the search cell
o  = json.load(open(s4.path4(f'v5_step4_{NAME}.json', 'res')))
an = json.load(open(s4.path4(f'v5_step4_analysis_{NAME}.json', 'an')))
tr = s4.chain()            # three-level transmons, N_tot <= 4
T_opt, n_best = o['T_opt'], o['n_opt']
Floquet = np.asarray(fv.floquet_operator(tr, np.atleast_2d(o['omegas_opt']), np.atleast_2d(o['couplings']),
                                         o['A_opt'], o['sigma_opt'], T_opt, fv.UNIT_PHYS, prec=fv.P64))
dim = Floquet.shape[0]
i_src, i_tgt = tr.src_i, tr.tgt_i
ket_source = np.zeros(dim, complex); ket_source[i_src] = 1.0
ket_target = np.zeros(dim, complex); ket_target[i_tgt] = 1.0
src_lbl, tgt_lbl = NAME.split('_')
print(f"{NAME} [step 4, three-level transmon, dim {dim}]: F={o['fidelity']:.6f} at n={n_best} | "
      f"A={o['A_opt']:.4f}, sigma={o['sigma_opt']:.4f}, T={T_opt:.4f}")

ns = np.arange(NPLOT + 1)
fids, p_src = np.empty(NPLOT + 1), np.empty(NPLOT + 1)
psi = ket_source.copy()
for n in ns:
    fids[n]  = abs(np.vdot(ket_target, psi))**2
    p_src[n] = abs(np.vdot(ket_source, psi))**2
    psi = Floquet @ psi
p_tgt = fids

out_txt = s4.path4(f'v5_step4_evolution_{NAME}.txt', 'an')
np.savetxt(out_txt, np.column_stack([ns, fids, p_src, p_tgt]), fmt=['%d', '%.8f', '%.8f', '%.8f'],
           header=f"3-site evolution (q1-c-q2), Gaussian pulse, {NAME}, step 4\n"
                  f"omegas={o['omegas_opt']} g={o['couplings']}\n"
                  f"A={o['A_opt']:.6f} sigma={o['sigma_opt']:.6f} T={T_opt:.4f} n_best={n_best}\n"
                  "n fidelity initial_population final_population")
print('Saved', out_txt)

plt.figure(figsize=(11, 5))
plt.plot(ns, fids, marker='o', markersize=3, linewidth=1, linestyle='--', label='with pulse')
if SHOW_STATIC:
    # same static Hamiltonian, no pulse, at the times t_n = n (T + 6 sigma)
    H0s = fv.build_H0(tr, np.atleast_2d(o['omegas_opt']), np.atleast_2d(o['couplings']), fv.UNIT_PHYS)[0]
    H0s = H0s.cpu().numpy() if hasattr(H0s, 'cpu') else np.asarray(H0s)
    Es, Vs = np.linalg.eigh(H0s)
    ws = Vs[i_tgt] * Vs[i_src]
    tn = ns[:XMAX + 1] * (T_opt + 6 * o['sigma_opt'])
    p_stat = np.abs(np.exp(-1j * np.outer(tn, Es)) @ ws) ** 2
    plt.plot(ns[:XMAX + 1], p_stat, color='gray', marker='s', markersize=2, linewidth=1, label='without pulse (A = 0)')
    plt.legend(fontsize=16)
    print(f"   without pulse: max over these times = {p_stat.max():.4f}")
plt.axvline(n_best, color='gray', ls=':', lw=1.5)
plt.xlabel('n (Stroboscopic Steps)', fontsize=22)
plt.ylabel(r'Fidelity $|\langle %s|%s\rangle|^2$' % (tgt_lbl, src_lbl), fontsize=22)
plt.grid(True); plt.xlim(0, XMAX); plt.xticks(fontsize=22); plt.yticks(fontsize=22)
plt.tight_layout()
if SAVE_PDF: plt.savefig(s4.path4(f'v5_step4_Fidelity_{NAME}.pdf', 'fig'), bbox_inches='tight')
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(ns, p_src, marker='o', markersize=3, linewidth=1, linestyle='--', label=f'Initial state population (|{src_lbl}⟩)')
plt.plot(ns, p_tgt, marker='o', markersize=3, linewidth=1, linestyle='--', label=f'Final state population (|{tgt_lbl}⟩)')
plt.axvline(n_best, color='gray', ls=':', lw=1.5)
plt.xlabel('n (stroboscopic steps)', fontsize=18); plt.ylabel('Population', fontsize=18)
plt.xlim(0, XMAX); plt.xticks(fontsize=22); plt.yticks(fontsize=22); plt.legend(fontsize=16); plt.grid(True)
plt.tight_layout()
if SAVE_PDF: plt.savefig(s4.path4(f'v5_step4_Populations_{NAME}.pdf', 'fig'), bbox_inches='tight')
plt.show()

In [ ]:
# === Noise analysis: F as a function of eta (robustness curve) ===
SHOW_MAX_N    = True      # adds F_max over n (metric of the original paper), dashed
MHZ_AXIS      = True      # top axis with the absolute error on the qubit frequencies, delta f_q [MHz]
ETA_REALISTIC = 1e-4      # shaded region (0 = none)
SAVE_PDF      = True
if 'mean_maxn' not in an['eta_scan']['levels'][0]:   # analysis saved before F_max over n existed: recompute
    print('recomputing the eta scan (adds F_max over n)...')
    an['eta_scan'] = s4.eta_scan4(o, eta_list=[l['eta'] for l in an['eta_scan']['levels']],
                                  n_samples=an['eta_scan']['n_samples'])
    json.dump(an, open(s4.path4(f'v5_step4_analysis_{NAME}.json', 'an'), 'w'), indent=1)
L = an['eta_scan']['levels']
eta = np.array([l['eta'] for l in L])
ket = r'$|%s\rangle\to|%s\rangle$' % (src_lbl, tgt_lbl)

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(eta, [l['mean'] for l in L], yerr=[l['std'] for l in L], fmt='o-', capsize=5, color='steelblue',
            label=r'$\bar F(n_{opt}=%d)$' % n_best)
if SHOW_MAX_N:
    ax.errorbar(eta, [l['mean_maxn'] for l in L], yerr=[l['std_maxn'] for l in L], fmt='s--', capsize=3,
                color='steelblue', alpha=0.55, ms=4, label=r'$\bar F_{\max}$ over $n$')
ax.axhline(L[0]['mean'], color='gray', ls='--', lw=1, label=f"Ideal fidelity = {L[0]['mean']:.4f}")
ax.set_xscale('symlog', linthresh=1e-6)
ax.set_xlim(-2e-7, eta.max() * 1.6)
if ETA_REALISTIC:
    ax.axvspan(0, ETA_REALISTIC, color='tab:green', alpha=0.08, lw=0,
               label=r'realistic static error ($\eta \leq 10^{%d}$)' % round(np.log10(ETA_REALISTIC)))
if MHZ_AXIS:
    k = np.mean([l['df_qubit_MHz'] / l['eta'] for l in L if l['eta'] > 0])
    top = ax.secondary_xaxis('top', functions=(lambda x: x * k, lambda y: y / k))
    top.set_xscale('symlog', linthresh=1e-6 * k)
    tk = [t for t in (0.01, 0.1, 1, 10) if t <= k * eta.max() * 1.6]
    top.set_xticks(tk, [f'{t:g}' for t in tk]); top.minorticks_off()
    top.set_xlabel(r'$\delta f_q$ [MHz]', fontsize=14)
    top.tick_params(labelsize=12)
ax.set_xlabel(r'Relative noise level  $\eta$', fontsize=20)
ax.set_ylabel(r'$\bar F$', fontsize=20)
ax.tick_params(axis='both', which='major', labelsize=15)
ax.set_title('Robustness curve ' + ket, fontsize=19)
ax.set_ylim(0, 1.05); ax.grid(True, which='both', alpha=0.3); ax.legend(fontsize=11, loc='lower left')
fig.tight_layout()
if SAVE_PDF: fig.savefig(s4.path4(f'v5_robustness_curve_step4_{NAME}.pdf', 'fig'), bbox_inches='tight')
plt.show()

print(f"\n{NAME}  (step 4, nominal F {o['fidelity']:.5f}, n_opt = {n_best})")
print(f"  {'eta':>8} {'df_q [MHz]':>11} {'<F> fixed n':>12} {'std':>8} {'p20':>8} {'min':>8} {'<F_max> over n':>15}")
for l in L:
    print(f"  {l['eta']:8.0e} {l['df_qubit_MHz']:11.3f} {l['mean']:12.5f} {l['std']:8.5f} {l['p20']:8.5f} "
          f"{l['min']:8.5f} {l['mean_maxn']:15.5f}")

In [ ]:
# === F vs T1 (Lindblad) ===
SAVE_PDF = True
S = an['lindblad']['T1_scan']
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot([s['T1_qubit_us'] for s in S], [s['F'] for s in S], 'o', ls='-', color='steelblue', label=ket + ' (Step 4)')
ax.set_xscale('log'); ax.set_xlabel(r'$T_1$ qubit [$\mu$s]  ($T_\varphi$ and coupler scaled together)', fontsize=15)
ax.set_ylabel(r'$F(n_{opt})$ with decoherence', fontsize=16)
ax.tick_params(labelsize=14); ax.grid(True, which='both', alpha=0.3); ax.legend(fontsize=10)
fig.tight_layout()
if SAVE_PDF: fig.savefig(s4.path4(f'v5_step4_F_vs_T1_{NAME}.pdf', 'fig'), bbox_inches='tight')
plt.show()
li = an['lindblad']
print(f"{NAME}: unitary F {li['F_unitary_check']:.5f} -> Lindblad {li['F_lindblad']:.5f}")

In [ ]:
# === F(n) +- std bands for each eta (one figure per eta) ===
ETA_TO_SHOW = (1e-5, 1e-4, 1e-3)
SAVE_PDF = True
for l in an['eta_scan']['levels']:
    if not any(np.isclose(l['eta'], e) for e in ETA_TO_SHOW):
        continue
    m, s = np.array(l['curve_mean']), np.array(l['curve_std'])
    n = np.arange(1, len(m) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(n, m, lw=2, label=r'$\bar F(n)$')
    ax.fill_between(n, m - s, m + s, alpha=0.3, label=r'$\pm\sqrt{\mathrm{Var}}$')
    ax.axvline(n_best, color='red', ls=':', lw=1.5, label=f'$n_{{opt}}={n_best}$')
    ax.set_xlabel('n (stroboscopic steps)', fontsize=16); ax.set_ylabel(r'$F(n)$', fontsize=16)
    ax.set_ylim(-0.05, 1.1); ax.grid(True, alpha=0.3); ax.legend(fontsize=11, loc='upper left')
    fig.tight_layout()
    eta_str = f"{l['eta']:.0e}".replace('-', 'm').replace('+', '')
    if SAVE_PDF: fig.savefig(s4.path4(f'v5_step4_band_{NAME}_eta{eta_str}.pdf', 'fig'), bbox_inches='tight')
    plt.show()

In [ ]:
# === Resolvent and quasi-energies (EXACT Floquet operator, finite-width kick) ===
from scipy.linalg import eig

T = T_opt + 2 * fv.WINDOW * o['sigma_opt']   # FULL Floquet period: pulse window (6 sigma) + free evolution T_opt
UF = Floquet
print(f"Floquet period T = T_opt + 6 sigma = {T:.4f} ns")

eigvals, eigvecs = eig(UF)
for j in range(dim):
    eigvecs[:, j] /= np.linalg.norm(eigvecs[:, j])

thetas = -np.angle(eigvals)
thetas = (thetas + np.pi) % (2 * np.pi) - np.pi
eps = thetas / T

istate, fstate = ket_source, ket_target
r = np.array([np.vdot(fstate, eigvecs[:, j]) * np.vdot(eigvecs[:, j], istate) for j in range(dim)])

def Gofz(z):
    return np.sum(r / (1 - z * eigvals))

wmin, wmax, npts = -np.pi / T, np.pi / T, 100000
wgrid = np.linspace(wmin, wmax, npts)
Gvals = np.array([Gofz(np.exp(-1j * w * T)) for w in wgrid])
Gmag  = np.abs(Gvals)

plt.figure(figsize=(9, 6))
plt.plot(wgrid, Gmag, 'b-', lw=2, label=r'$|G(e^{-i\omega T})|$')
for i, e in enumerate(eps):
    plt.axvline(e, color='crimson', linestyle='--', lw=1, alpha=0.5,
                label=r'$\epsilon_\alpha$' if i == 0 else None)
plt.xlabel(r'$\omega$', fontsize=12); plt.ylabel(r'$|G(e^{-i\omega T})|$', fontsize=12)
plt.title(f'Floquet channels and quasi-energy poles (exact Gaussian kick) — {NAME}', fontsize=13)
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(s4.path4(f'v5_step4_Resolvent_{NAME}.pdf', 'fig'), bbox_inches='tight'); plt.show()

np.savetxt(s4.path4(f'v5_step4_floquet_eps_{NAME}.dat', 'an'), eps)
np.savetxt(s4.path4(f'v5_step4_floquet_eps_{NAME}_sorted.dat', 'an'), np.sort(eps))

unitarity_error = np.linalg.norm(UF.conj().T @ UF - np.eye(dim), np.inf)
print("Unitarity error (exact UF):", unitarity_error)
print("Dominant residues:", np.sort(np.abs(r))[::-1][:10])

In [ ]:
# === Simple FFT of the fidelity (omega basis) ===
fidelity_data = np.array(fids)
N_data = len(fidelity_data)

fft_result = np.fft.fft(fidelity_data)
amplitude_spectrum = np.abs(fft_result) / N_data

freq_axis_nu = np.fft.fftfreq(N_data, d=1.0)
fft_shifted_nu = np.fft.fftshift(amplitude_spectrum)
freq_shifted_nu = np.fft.fftshift(freq_axis_nu)
omega_fft_axis = (2 * np.pi / T) * freq_shifted_nu   # T = full period (resolvent cell)

plt.figure(figsize=(10, 5))
plt.plot(omega_fft_axis, fft_shifted_nu, 'b-', lw=2)

fft_shifted_display = fft_shifted_nu.copy()
fft_shifted_display[np.argmin(np.abs(omega_fft_axis))] = 0.0
peak_indices = np.argsort(fft_shifted_display)[::-1][:3]
peak_omegas, peak_amps = omega_fft_axis[peak_indices], fft_shifted_nu[peak_indices]
for omega, amp in zip(peak_omegas, peak_amps):
    if amp > 0.01:
        plt.scatter(omega, amp, color='red', marker='o')
        plt.annotate(f'{amp:.3f}\n({omega:.4f})', (omega, amp),
                     textcoords="offset points", xytext=(5, -10), ha='center')

plt.title(r'Floquet frequency spectrum of the fidelity ($\omega$ basis)')
plt.xlabel(r'Quasi-energy $\omega$', fontsize=12); plt.ylabel('Amplitude (Fourier spectrum)', fontsize=12)
plt.grid(True); plt.xlim(-np.pi / T, np.pi / T); plt.tight_layout(); plt.show()

print("\n--- Spectrum analysis ---")
for omega, amp in sorted(zip(peak_omegas, peak_amps), key=lambda x: x[1], reverse=True):
    if amp > 0.01:
        print(f"  omega: {omega:.5f}, amplitude: {amp:.5f}")

In [ ]:
# === Windowed FFT + comparison with the quasi-energies eps ===
from scipy.signal import find_peaks, windows

signal = np.array(fids)
N = signal.size
T = T_opt + 2 * fv.WINDOW * o['sigma_opt']   # full Floquet period (6 sigma pulse window + free evolution)

sig_win = signal * windows.hann(N)
pad_factor = 8
Npad = int(N * pad_factor)
sig_pad = np.pad(sig_win, (0, Npad - N), mode='constant')

S = np.fft.fft(sig_pad)
omega_axis = 2 * np.pi * np.fft.fftfreq(Npad, d=T)
S_mag = np.abs(S)
omega_axis_shifted = np.fft.fftshift(omega_axis)
S_mag_shifted = np.fft.fftshift(S_mag)

half = Npad // 2
omega_pos, Spos = omega_axis_shifted[half:], S_mag_shifted[half:]
peaks, props = find_peaks(Spos, height=np.max(Spos) * 0.05, distance=3)
peak_omegas, peak_heights = omega_pos[peaks], props['peak_heights']

eps_wrapped = ((eps + np.pi / T) % (2 * np.pi / T)) - np.pi / T
matches = []
for om in peak_omegas:
    i_c = np.argmin(np.abs(eps_wrapped - om))
    matches.append((om, eps_wrapped[i_c], eps[i_c], np.abs(eps_wrapped[i_c] - om)))

print("Angular resolution Δω =", 2 * np.pi / (Npad * T))
print("Peaks found (omega, matched_eps_wrapped, original_eps, distance):")
for m in matches:
    print(f"{m[0]:.6e}   {m[1]:.6e}   {m[2]:.6e}   diff = {m[3]:.3e}")

plt.figure(figsize=(9, 4))
plt.plot(omega_axis_shifted, S_mag_shifted, label='|FFT(signal)| (shifted)')
for e in eps_wrapped:
    plt.axvline(e, color='crimson', linestyle='--', alpha=0.6)
plt.xlim(-np.pi / T, np.pi / T)
plt.xlabel(r'$\omega$ (rad/ns)'); plt.ylabel('FFT amplitude')
plt.title('Fidelity spectrum (FFT) and quasi-energy lines')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# === Delta-epsilon transitions weighted by the residues, overlaid on the spectrum ===
c = np.array([np.vdot(eigvecs[:, a], ket_source) for a in range(dim)])
d = np.array([np.vdot(ket_target, eigvecs[:, a]) for a in range(dim)])

DE_mat = eps[:, None] - eps[None, :]
DE_mat = ((DE_mat + np.pi / T) % (2 * np.pi / T)) - np.pi / T
WE_mat = np.abs((c[:, None] * np.conj(c)[None, :]) * (d[None, :] * np.conj(d)[:, None]))
DE, WE = DE_mat.ravel(), WE_mat.ravel()
pairs = [(a, b) for a in range(dim) for b in range(dim)]

topK = 50
top_idx = np.argsort(WE)[::-1][:topK]
print("Top transitions (a,b) with wrapped frequency and weight:")
for ii in top_idx:
    a, b = pairs[ii]
    print(f"({a},{b})  Δε_wrapped = {DE[ii]:.6e}   weight = {WE[ii]:.3e}")

plt.figure(figsize=(11, 5))
for ii in top_idx:
    plt.axvline(DE[ii], color='red', linestyle='--', linewidth=1, alpha=0.6)
plt.plot(omega_axis_shifted, S_mag_shifted, lw=1, color='tab:blue')
plt.xlim(-np.pi / T, np.pi / T)
plt.xlabel(r'$\omega$ (rad/ns)', fontsize=20); plt.ylabel('FFT Amplitude', fontsize=20)
plt.grid(alpha=0.3)
plt.gca().ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
plt.xticks(fontsize=18); plt.yticks(fontsize=18); plt.tight_layout()
plt.savefig(s4.path4(f'v5_step4_comparison_{NAME}.pdf', 'fig'), bbox_inches='tight'); plt.show()